# Active learning for astronomical transients
### A CPU-only AppleCiDEr laboratory

In the first notebook, we chose which handwritten digits to label. Now we will ask the same question about astronomical transients: **which object should we investigate next when labels are expensive?**

We will follow two complementary investigations:

- **Part A — Run an active-learning experiment.** Explore astronomical cutouts and object-level features, train a small CPU classifier, select objects with four acquisition policies, reveal their labels through an oracle, and measure what changes.
- **Part B — Examine an astronomical research experiment.** Compare our simplified loop with saved validation results from completed AstroMiNN training runs. We can analyze those records on a CPU, but they are not a substitute for retraining the original model.

No GPU or pretrained checkpoint is required. The notebook runs with NumPy, scikit-learn, Matplotlib and the provided `al_pipeline` package.

> **Check the data banner in the next cell.** Until a reviewed astronomical export is installed, Part A uses a clearly marked **synthetic demonstration**. Its results are useful for understanding the mechanics, but are not measurements of ZTF classification performance. Part B uses separately identified historical measurements; it never substitutes synthetic results for missing experimental records.


## 1. Load the data and check its origin

First, identify where the features came from. The notebook uses a small local file when one has been supplied; otherwise, it creates a synthetic example to practice the workflow. The banner and provenance printed below tell you which case you are viewing.

All calculations run on CPUs. You can change `USE_PRESENTATION_STYLE` to control the figure appearance without changing the experiment.


In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display, Markdown
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score

from al_pipeline.astronomy.teaching_data import CLASS_NAMES, make_synthetic_demo, load_teaching_data
from al_pipeline.core.batch_selection import DiversitySelector
from al_pipeline.core.evaluation.history import EvaluationHistory
from al_pipeline.core.evaluation.metrics import compute_classification_metrics
from al_pipeline.core.evaluation.plotting import plot_strategy_comparison
from al_pipeline.core.evaluation.notebook_style import use_notebook_style, policy_color, polish_policy_axes
from al_pipeline.core.loop import ActiveLearningLoop
from al_pipeline.core.oracle.existing_labels import ExistingLabelsOracle
from al_pipeline.core.pool import Pool
from al_pipeline.core.scoring.random import RandomScorer
from al_pipeline.core.scoring.entropy import EntropyScorer
from al_pipeline.core.scoring.margin import MarginScorer
from al_pipeline.core.selection import TopKSelector
from al_pipeline.core.types import ClassificationPredictions

# Embed figures in GitHub's notebook preview, including when executing headlessly.
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')
USE_PRESENTATION_STYLE = True
if USE_PRESENTATION_STYLE:
    use_notebook_style()

# Supports both starting Jupyter in the repository root and in examples/.
DATA_DIR = next((d for d in (Path('examples/data'), Path('data')) if d.is_dir()), Path('examples/data'))
REAL_DATA = DATA_DIR / 'astro_real_v1.npz'
SEED = 135
INITIAL_PER_CLASS = 4
BATCH_SIZE = 20
N_CYCLES = 6
POLICIES = ('random', 'entropy', 'margin', 'entropy_diversity')
METRICS = ('f1_macro', 'balanced_accuracy')

if REAL_DATA.exists():
    dataset = load_teaching_data(REAL_DATA)
    if 'synthetic' in str(dataset['provenance']).lower():
        raise ValueError('A synthetic fixture cannot be advertised as a real astronomical export')
    REAL_ASTRONOMY = True
else:
    dataset = make_synthetic_demo(seed=SEED)
    REAL_ASTRONOMY = False

banner = ('REAL ASTRONOMICAL EXPORT' if REAL_ASTRONOMY else
          '⚠ SYNTHETIC DEMONSTRATION — NOT ZTF DATA / NOT ASTROMINN RESULTS')
display(Markdown('**' + banner + '**'))
print('Feature provenance:', str(dataset['provenance']))
print('Representation:', str(dataset['feature_source']))

### Why do we work with objects rather than individual alerts?

A transient may generate several alerts as it evolves. If we acquired individual alerts independently, we could accidentally request multiple labels for the same object—and even put alerts from one object into both training and evaluation.

Instead, **one object is one acquisition unit**. The compact dataset retains one feature vector per object (derived from its latest available alert), the number of its available alerts, and fixed train/validation/test memberships. The labels of candidate objects belong to the simulated oracle, not to the acquisition scorer.

For the real export, the features summarize the science, reference and difference image channels on $8\times8$ grids, together with 34 metadata values. This is a simple CPU-ready representation, **not** a pretrained AstroMiNN embedding. After our first experiment, we can ask what would change with better representations.


In [ ]:
X = dataset['features']
y = dataset['labels']  # used only for partitioning, evaluation and a simulated oracle
object_ids = dataset['object_ids']
splits = dataset['splits']
alert_counts = dataset['alert_counts']
rows = {name: np.flatnonzero(splits == name) for name in ('train','validation','test')}
id_to_row = {str(oid): int(i) for i, oid in enumerate(object_ids)}
assert len(rows['train']) + len(rows['validation']) + len(rows['test']) == len(y)
assert len(set(object_ids)) == len(object_ids)
assert set(np.unique(y[rows['train']])) == set(range(len(CLASS_NAMES)))

split_table = pd.DataFrame({name: np.bincount(y[indices], minlength=len(CLASS_NAMES))
                           for name, indices in rows.items()}, index=CLASS_NAMES)
display(split_table.rename_axis('Target class'))
print('Objects:',len(y),' | feature dimension:',X.shape[1],
      '| median alerts/object:',int(np.median(alert_counts)))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 3.4))
axes[0].bar(CLASS_NAMES, split_table['validation'], color=policy_color('entropy'))
axes[0].set(title='Class support in the fixed validation set', ylabel='Objects')
axes[1].hist(alert_counts[rows['train']], bins=np.arange(1, 13)-0.5,
             color=policy_color('margin'), edgecolor='white')
axes[1].set(title='Available alert counts (training population)', xlabel='Alerts per object', ylabel='Objects')
for ax in axes: ax.grid(axis='y', alpha=.35)
plt.tight_layout()
plt.show()

## 2. Inspect the astronomical images

Each transient sample contains a **science** image of the observed field, a **reference** image representing a baseline, and a **difference** image that highlights changes between observations. Compare their appearance across the five classes.

The gallery uses a few *validation* examples, where labels are available for illustration. Those labels do not enter the training pool or influence acquisition. If the banner says *synthetic*, these are artificial cutouts rather than observed ZTF sources.


In [ ]:
thumbs = dataset['preview_images']
preview_ix = dataset['preview_indices']
fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(7.8, 9))
for class_index, name in enumerate(CLASS_NAMES):
    matches = [j for j, idx in enumerate(preview_ix) if int(y[idx]) == class_index]
    if not matches: continue
    sample = thumbs[matches[0]]
    for channel, ax in enumerate(axes[class_index]):
        v = sample[channel]
        p1, p99 = np.percentile(v, [2, 98])
        ax.imshow(v, cmap='gray_r', vmin=p1, vmax=max(p99, p1 + .001))
        ax.set_xticks([]); ax.set_yticks([])
        if channel == 0: ax.set_ylabel(name, rotation=0, labelpad=26, va='center')
        if class_index == 0: ax.set_title(('Science', 'Reference', 'Difference')[channel])
fig.suptitle('Sample cutout triplets — ' + ('historical images' if REAL_ASTRONOMY else 'SYNTHETIC illustration'))
plt.tight_layout()
plt.show()

## 3. Begin with a small labeled set, $L_0$

We start with four confirmed examples of each class. This deliberately balanced initialization makes it possible to train a five-class classifier; it does **not** represent the true abundance of the classes in the sky.

All other training objects begin in the unlabeled pool, $U_0$. The validation set will measure progress after each cycle, while the test set stays untouched until the last section.

A **simulated oracle** keeps the withheld training labels. We may request a label only *after* choosing an eligible object. Watch how the pool changes from unlabeled to pending to labeled.


In [ ]:
train_rows = rows['train']
rng = np.random.default_rng(SEED)
initial_rows = np.concatenate([
    rng.choice(train_rows[y[train_rows] == cls], size=INITIAL_PER_CLASS, replace=False)
    for cls in range(len(CLASS_NAMES))
])
initial_ids = tuple(str(object_ids[i]) for i in initial_rows)
initial_labels = {str(object_ids[i]): int(y[i]) for i in initial_rows}
train_ids = tuple(str(object_ids[i]) for i in train_rows)
# The following labels belong to the *oracle*, not the acquisition model.
oracle = ExistingLabelsOracle({str(object_ids[i]): int(y[i]) for i in train_rows})
assert len(initial_ids) == len(CLASS_NAMES)*INITIAL_PER_CLASS == 20
assert set(initial_ids).issubset(set(train_ids))
print('L0 class mix:', {CLASS_NAMES[c]: list(initial_labels.values()).count(c)
                         for c in range(len(CLASS_NAMES))})
print('Candidate pool size:', len(train_ids)-len(initial_ids))

### First fit: train only on labels we already have

Our classifier is intentionally small: standardized features followed by logistic regression. At each cycle we refit it using the labels currently in $L_t$. The feature scaler is fitted on $L_t$ as well—not on hidden candidates or evaluation objects.

This is a teaching model, not AstroMiNN. The purpose is to observe how **which objects we label** changes the data available for the next training round.


In [ ]:
def fit_classifier(observed):
    indices = [id_to_row[oid] for oid in sorted(observed)]
    labels = np.asarray([observed[oid] for oid in sorted(observed)], dtype=int)
    assert set(np.unique(labels)) == set(range(len(CLASS_NAMES)))
    model = make_pipeline(StandardScaler(), LogisticRegression(
        C=0.15, max_iter=250, solver='lbfgs', class_weight='balanced'))
    model.fit(X[indices], labels)
    return model


def validation_scores(model):
    pred = ClassificationPredictions(model.predict_proba(X[rows['validation']]),
                                     output_type='probabilities')
    return compute_classification_metrics(y[rows['validation']], pred,
                                          metrics=METRICS, labels=tuple(range(len(CLASS_NAMES))))

baseline = fit_classifier(initial_labels)
display(pd.DataFrame([validation_scores(baseline)]).round(3))

## 4. Repeat the acquisition cycle

At each cycle, we train the classifier, predict probabilities for eligible unlabeled objects, assign acquisition scores, select a batch, request its labels, and update the pool.

We compare **random**, **entropy**, **margin**, and **entropy + diversity** selection. They all begin with the same $L_0$, use the same labeling budget, and are evaluated on the same validation objects.

Entropy rewards spread-out class probabilities; margin favors close top-two predictions. Entropy + diversity first finds uncertain candidates and then spreads a batch across feature space. This is **diversity within one batch**, rather than full coverage of the labeled population.

Notice that the model never sees the true class of a candidate before acquisition. Only the oracle reveals it.


In [ ]:
def run_policy(policy, *, seed=SEED):
    observed = dict(initial_labels)
    pool = Pool.from_ids(train_ids, labeled_ids=initial_ids)
    history = EvaluationHistory()
    acquired_batches, acquired_labels = [], []
    generator = np.random.default_rng(seed)
    scorer = {'random': RandomScorer(generator), 'entropy': EntropyScorer(),
              'margin': MarginScorer(), 'entropy_diversity': EntropyScorer()}[policy]
    for cycle in range(N_CYCLES + 1):
        assert set(observed) == set(pool.labeled_ids)
        model = fit_classifier(observed)
        measures = validation_scores(model)
        history.add(cycle=cycle, n_labeled=len(observed),
                    n_acquired=0 if cycle == 0 else len(acquired_batches[-1]),
                    split='validation', metrics=measures,
                    strategy=policy, seed=seed, run_id=f'{policy}-{seed}')
        if cycle == N_CYCLES or not pool.unlabeled_ids:
            break
        candidates = sorted(pool.unlabeled_ids)
        candidate_rows = np.asarray([id_to_row[oid] for oid in candidates], dtype=int)
        predictions = (len(candidates) if policy == 'random' else
                       ClassificationPredictions(model.predict_proba(X[candidate_rows]),
                                                 output_type='probabilities'))
        selector = (DiversitySelector(X[candidate_rows], shortlist_multiplier=3.0)
                    if policy == 'entropy_diversity' else TopKSelector())
        selection = ActiveLearningLoop(scorer=scorer, selector=selector, pool=pool).acquire(
            ids=candidates, acquisition_data=predictions, k=min(BATCH_SIZE, len(candidates)))
        assert set(selection.ids).issubset(pool.pending_ids)
        assert not set(selection.ids).intersection(observed)
        received = oracle.query(selection)
        for oid, label in zip(received.ids, received.labels):
            observed[oid] = int(label)
        pool.resolve(received.ids)
        assert not pool.pending_ids
        acquired_batches.append(tuple(selection.ids))
        acquired_labels.append(tuple(int(z) for z in received.labels))
    return {'history':history, 'model':model, 'observed':observed, 'pool':pool,
            'batches':acquired_batches, 'revealed':acquired_labels}

runs = {policy: run_policy(policy) for policy in POLICIES}
expected = len(initial_ids) + N_CYCLES * BATCH_SIZE
assert all(run['history'].records[-1].n_labeled == expected for run in runs.values())
assert all(not run['pool'].pending_ids for run in runs.values())
summary = pd.DataFrame([{
    'policy': policy,
    'initial macro-F1': run['history'].records[0].metrics['f1_macro'],
    'final macro-F1': run['history'].records[-1].metrics['f1_macro'],
    'final balanced accuracy': run['history'].records[-1].metrics['balanced_accuracy'],
    'new labels': expected-len(initial_ids)
} for policy, run in runs.items()])
display(summary.round(3))
print('Same initial L0, same budgets, separate pool/oracle transitions for every policy.')

## 5. What did the new labels buy us?

A useful policy should improve predictions **early**, not merely achieve a high score at the final cycle. First, compare validation macro-F1 and balanced accuracy against the *number of labeled objects*. This makes the shared labeling budget visible.

Then inspect which classes each policy actually acquired. Those counts can be computed only **after** the oracle returns labels. Do uncertainty sampling and diversity change the composition of the labeled population?


In [ ]:
combined = EvaluationHistory([record for run in runs.values()
                              for record in run['history'].records])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(axes, METRICS, ('Validation macro-F1','Validation balanced accuracy')):
    plot_strategy_comparison(combined, metric, strategies=POLICIES,
                             split='validation', x_axis='n_labeled', ax=ax)
    ax.set_title(title); ax.set_ylabel(title)
    polish_policy_axes(ax)
fig.suptitle('Interactive CPU classifier — ' + ('real astronomical sample' if REAL_ASTRONOMY else 'SYNTHETIC EXAMPLE'))
plt.tight_layout()
plt.show()

display(pd.DataFrame([{
    'policy': policy, 'cycle': record.cycle, 'n_labeled': record.n_labeled,
    'macro_F1': record.metrics['f1_macro']}
    for policy, run in runs.items() for record in run['history'].records
]).pivot(index='n_labeled', columns='policy', values='macro_F1').round(3))

In [ ]:
fig, axes = plt.subplots(1, len(POLICIES), figsize=(13.2, 3.3), sharey=True)
for ax, policy in zip(axes, POLICIES):
    labels = [c for batch in runs[policy]['revealed'] for c in batch]
    hist = np.bincount(labels, minlength=len(CLASS_NAMES))
    ax.bar(range(len(CLASS_NAMES)), hist, color=policy_color(policy), edgecolor='white')
    ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha='right')
    ax.set_title(policy.replace('_',' + ')); ax.grid(axis='y', alpha=.25)
axes[0].set_ylabel('Revealed acquisition labels')
fig.suptitle('Class mixture of selected follow-up objects')
plt.tight_layout()
plt.show()

acquired = {p:set(oid for batch in run['batches'] for oid in batch)
            for p, run in runs.items()}
intersection = pd.DataFrame([
    [len(acquired[left] & acquired[right]) for right in POLICIES]
    for left in POLICIES], index=POLICIES, columns=POLICIES)
display(intersection)
print('Acquisition overlap counts exclude L0. Shared objects do not imply identical learning curves.')

### Look beyond overall accuracy

Astronomical classes can have very different numbers of examples. Macro-F1 gives each class equal weight, but it does not show *which* class improved. Use the fixed validation set to examine per-class recall.

Pay particular attention to rare classes such as TDE. A small validation support means that one correctly classified object may change recall substantially, so always read the support counts alongside the plotted values.


In [ ]:
recall_rows = []
for policy, run in runs.items():
    pred = run['model'].predict(X[rows['validation']])
    recall = recall_score(y[rows['validation']], pred,
                          labels=np.arange(len(CLASS_NAMES)), average=None, zero_division=0)
    for label, score in zip(CLASS_NAMES, recall):
        recall_rows.append({'policy':policy, 'class':label, 'recall':float(score)})
recall_df = pd.DataFrame(recall_rows)
display(recall_df.pivot(index='class', columns='policy', values='recall').round(3))
fig, ax = plt.subplots(figsize=(9.0, 3.8))
for j, policy in enumerate(POLICIES):
    vals = recall_df[recall_df.policy == policy]['recall'].values
    ax.bar(np.arange(len(CLASS_NAMES)) + (j-(len(POLICIES)-1)/2)*.18,
           vals, width=.18, color=policy_color(policy), label=policy)
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set(ylim=(0,1), ylabel='Validation recall', title='Per-class recall of the final CPU classifiers')
ax.legend(ncol=2); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

## 6. What happened in real AstroMiNN active-learning runs?

So far, we have trained and retrained a **CPU surrogate**. Next, we switch from running an experiment to **reading measurements from a different experiment** that used the AstroMiNN architecture on historical astronomical data.

Start with the reported comparison below. The independent full-data model provides a reference; the BALD and TAILOR entries are results recorded at an earlier acquisition cycle. Ask: *Can a model trained with fewer labeled objects outperform a reference trained on the full labeled set? What happens to rare-class recall and false positives?*

These values come from the AppleCiDEr research presentation (November 2026). They are **separate from Part A**: the architecture, feature representations, training procedure and data budgets differ. Do not compare their raw macro-F1 values as if they were controlled runs of one experiment. Saved summaries also cannot tell us how AstroMiNN would respond to a new, untested acquisition policy.


In [ ]:
# Reported point estimates from the presentation; NOT reconstructed learning curves.
reported = pd.DataFrame([
    {'model':'Full-data AstroMiNN', 'training fraction':1.0, 'macro-F1':.6367,
     'TDE F1':.194, 'TDE recall':.706, 'TDE false positives':95.7},
    {'model':'BALD at cycle 2', 'training fraction':.30, 'macro-F1':.6534,
     'TDE F1':.265, 'TDE recall':.510, 'TDE false positives':41.0},
    {'model':'TAILOR at cycle 2', 'training fraction':.30, 'macro-F1':.6516,
     'TDE F1':.288, 'TDE recall':.627, 'TDE false positives':46.7},
])
display(reported)
fig, axes = plt.subplots(1,2,figsize=(12,3.7))
bar_colors = ['#B7A588', '#83B6CD', '#AE9DC7']
for ax, metric, title in zip(axes, ('macro-F1','TDE F1'),
                             ('Validation macro-F1', 'TDE F1 (small support)')):
    ax.bar(reported.model, 100*reported[metric], color=bar_colors, width=.58)
    ax.set(title=title, ylabel='Percent')
    ax.tick_params(axis='x', labelrotation=18)
    ax.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.show()
print('Source: supplied Final.pdf, acquisition-policy findings and rare-class performance slides.')
print('These are *different* model-training recipes and labeled fractions, not a causal comparison.')

### Read the complete acquisition history, if available

A few reported points are useful, but they do not show how performance changes *between* cycles or how results vary across training seeds. Here we can load a small archive of **recorded** validation measurements and plot their learning curves.

The archive is optional because it is prepared separately from the model training. Without it, the notebook displays only the reported comparison above—**it does not create fictitious intermediate cycles**.

- `astro_historical_evaluation.csv` contains validation metrics by policy, seed and cycle.
- Optional acquisition, overlap and per-class CSVs allow further analysis of class recovery and selected populations.

When the archive is available, compare early label efficiency, endpoint performance and variation across seeds. Remember that the plot shows completed experiments, not a new run of AstroMiNN.


In [ ]:
real_replay_path = DATA_DIR / 'astro_historical_evaluation.csv'
if not real_replay_path.exists():
    print('Full historical replay not yet distributed. The table above is genuine but limited to reported points.')
    print('When reviewed historical tables are added, this section can display the complete recorded learning curves.')
    historical_df = None
else:
    historical_df = pd.read_csv(real_replay_path)
    required = {'suite','initialization','strategy','seed','cycle','split',
                'metric','value','n_labeled','n_acquired_total'}
    assert required.issubset(historical_df.columns), 'Historical export missing required columns'
    historical_df = historical_df[
        (historical_df['split'] == 'validation') &
        (historical_df['metric'].isin(['f1_macro','balanced_accuracy']))].copy()
    assert len(historical_df), 'Export lacks validation macro-F1/balanced accuracy'
    assert not historical_df.duplicated(['suite','initialization','strategy','seed','cycle','metric']).any()
    # Explicit choice avoids silently merging incompatible experiment suites.
    selected_suite = sorted(historical_df['suite'].unique())[0]
    selected_init = sorted(historical_df[historical_df.suite == selected_suite]['initialization'].unique())[0]
    subset = historical_df[(historical_df.suite == selected_suite) &
                           (historical_df.initialization == selected_init) &
                           (historical_df.metric == 'f1_macro')]
    stats = (subset.groupby(['strategy','cycle'],as_index=False)
             .agg(macro_F1=('value','mean'), sd=('value','std'),
                  n_labeled=('n_labeled','mean'), n_seeds=('seed','nunique')))
    fig, ax = plt.subplots(figsize=(9.5,4.3))
    for policy, group in stats.groupby('strategy'):
        group = group.sort_values('n_labeled')
        x = group.n_labeled.to_numpy(dtype=float)
        avg = group.macro_F1.to_numpy(dtype=float)
        sd = group.sd.fillna(0).to_numpy(dtype=float)
        ax.plot(x,avg,marker='o',label=str(policy),color=policy_color(str(policy)))
        ax.fill_between(x,avg-sd,avg+sd,alpha=.15,color=policy_color(str(policy)))
    ax.set(title=f'Recorded AstroMiNN trajectories: {selected_suite} / {selected_init}',
           xlabel='Cumulative labeled objects', ylabel='Validation macro-F1')
    ax.legend(ncol=2); ax.grid(axis='y',alpha=.3)
    plt.tight_layout();plt.show()
    display(stats.round(3).head(18))
    print('These curves belong to completed AstroMiNN training runs, not the CPU model above.')

In [ ]:
if historical_df is not None:
    extra = DATA_DIR / 'astro_historical_acquisition.csv'
    if extra.exists():
        acquired_history = pd.read_csv(extra)
        subset = acquired_history[(acquired_history.suite == selected_suite) &
                                  (acquired_history.initialization == selected_init)]
        last_cycle = subset['cycle'].max()
        final = subset[subset.cycle == last_cycle]
        composition = (final.groupby(['strategy','label'])['cumulative_count'].mean()
                       .unstack(fill_value=0))
        display(composition.round(1))
        ax = composition.plot(kind='bar',stacked=True,figsize=(9.0,3.6),colormap='Pastel1')
        ax.set(title=f'Historical acquired labels by cycle {last_cycle} (mean across seeds)',
               ylabel='Cumulative acquired objects',xlabel='Policy')
        ax.legend(title='Revealed class',bbox_to_anchor=(1,1))
        plt.tight_layout();plt.show()
    else:
        print('Optional acquisition-composition CSV not installed.')

## 7. Reflect and try a new question

We have now seen two different uses of active learning: **running a lightweight closed loop** and **analyzing completed research trajectories**. Both depend on consistent object IDs, careful pool transitions, fixed evaluation splits and an explicit labeling budget.

**Optional investigations — choose one to implement yourself:**

1. **More or fewer alerts?** Compare the alert counts of acquired objects across policies. Does uncertainty favor objects with more observations? What changes when you restrict eligibility to objects with at most ten alerts?
2. **Rare-class acquisition.** Examine how many TDE and CV objects are acquired early. Design an exploration rule using only *confirmed* labels, never hidden candidate labels.
3. **Delayed labels.** Leave some requests in the pending state for a cycle. How should the next training set and remaining budget change?
4. **Different features.** If a safely pretrained or $L_0$-only representation is later provided, rerun the same CPU classifier. Compare its learning curve with the simple image/metadata representation. Why would training a feature extractor on hidden test labels invalidate the comparison?
5. **Historical variability.** With the archived CSVs installed, compare early improvement, class recovery and variation across seeds. Does the policy with the highest endpoint always improve first?

For each exercise, state which information is visible at acquisition time and which is revealed only for evaluation.


## 8. Keep the test set sealed until the end

There is one final step for **Part A only**. Select a policy using its last **validation** macro-F1, then measure the chosen classifier on the untouched test set exactly once.

The historical AstroMiNN section uses its own recorded evaluation results and never reads this notebook's test labels. The test score below evaluates the selected CPU teaching classifier; it is not a published AstroMiNN result.


In [ ]:
winner = sorted(POLICIES, key=lambda p: (-runs[p]['history'].records[-1].metrics['f1_macro'], p))[0]
final_model = runs[winner]['model']
test_pred = ClassificationPredictions(final_model.predict_proba(X[rows['test']]),
                                      output_type='probabilities')
test_metrics = compute_classification_metrics(y[rows['test']], test_pred,
                                               metrics=METRICS,
                                               labels=tuple(range(len(CLASS_NAMES))))
print('Selected on validation:',winner)
print('ONE sealed-test score:',{key:round(value,3) for key,value in test_metrics.items()})
if not REAL_ASTRONOMY:
    print('Reminder: all Part A scores above come from synthetic class features, not ZTF.')